# ICLR Results

Plots the W&B groups tagged `ICLR`, using log paths relative to the `MyExpoComm` working directory. Runs in each group are aggregated in the same way as `reproduce_results.ipynb`.

# Adversarial Pursuit

The plotting function below is the adversarial-pursuit function from `reproduce_results.ipynb`.

In [ ]:
import re
import os
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

def parse_group_logs(file_paths):
    aggregated_data = defaultdict(list)
    t_env_pattern = re.compile(r"Recent Stats \| t_env:\s*(\d+)")
    test_return_pattern = re.compile(r"test_return_mean:\s*(-?\d+\.?\d*)")

    for path in file_paths:
        if not os.path.exists(path):
            print(f"Warning: File not found at {path}. Skipping from group configuration.")
            continue
        current_t_env = None
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if "Recent Stats | t_env:" in line:
                    t_match = t_env_pattern.search(line)
                    if t_match:
                        current_t_env = int(t_match.group(1))
                if current_t_env is not None and "test_return_mean:" in line:
                    r_match = test_return_pattern.search(line)
                    if r_match:
                        aggregated_data[current_t_env].append(float(r_match.group(1)))
                        current_t_env = None

    sorted_timesteps = sorted(aggregated_data)
    means = [np.mean(aggregated_data[t]) for t in sorted_timesteps]
    mins = [np.min(aggregated_data[t]) for t in sorted_timesteps]
    maxes = [np.max(aggregated_data[t]) for t in sorted_timesteps]
    return sorted_timesteps, np.array(means), np.array(mins), np.array(maxes)

def plot_multi_group_performance_advP(groups_data, plot_title="Mean Test Return"):
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=(11, 6.5))
    colors = ["#FF0000", "#0000FF", "#137333"]

    for idx, (group_name, paths) in enumerate(groups_data.items()):
        steps, y_mean, y_min, y_max = parse_group_logs(paths)
        if not steps:
            print(f"Skipping plot for group '{group_name}' due to missing data.")
            continue
        color = colors[idx % len(colors)]
        num_runs = len(paths)
        if num_runs > 1:
            ax.fill_between(steps, y_min, y_max, color=color, alpha=0.15,
                            label=f"{group_name} Range (Min/Max)")
        ax.plot(steps, y_mean, color=color, linewidth=2.5,
                label=f"{group_name} Mean ({num_runs} runs)")

    ax.set_title(plot_title, fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Timestep", fontsize=11, labelpad=10)
    ax.set_ylabel("Mean Test Return", fontsize=11, labelpad=10)
    ax.ticklabel_format(style="scientific", scilimits=(0, 0), axis="x")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(frameon=True, facecolor="white", edgecolor="none", loc="best")
    plt.tight_layout()
    plt.show()

In [ ]:
ADVP_EXPOCOMM = [
    "logs/advP_EC_op_nomm_42_2143352.err",
    "logs/advP_EC_op_nomm_43_2143358.err",
    "logs/advP_EC_op_nomm_44_2143362.err",
]
ADVP_HYBRIDCOMM = [
    "logs/advP_HC_op_nomm_42_2143357.err",
    "logs/advP_HC_op_nomm_43_2143359.err",
    "logs/advP_HC_op_nomm_44_2143363.err",
]
ADVP_IQL_RNN = [
    "logs/advP_IQL_RNN_nomm_42_2143355.err",
    "logs/advP_IQL_RNN_nomm_43_2143360.err",
    "logs/advP_IQL_RNN_nomm_44_2143364.err",
]
ADVP_BIG_EXPOCOMM = [
    "logs/advP_B_EC_op_nomm_42_2147127.err",
    "logs/advP_B_EC_op_nomm_43_2147130.err",
    "logs/advP_B_EC_op_nomm_44_2147133.err",
]
ADVP_BIG_HYBRIDCOMM = [
    "logs/advP_B_HC_op_nomm_42_2147128.err",
    "logs/advP_B_HC_op_nomm_43_2147131.err",
    "logs/advP_B_HC_op_nomm_44_2147134.err",
]
ADVP_BIG_IQL_RNN = [
    "logs/advP_B_IQL_RNN_nomm_42_2147129.err",
    "logs/advP_B_IQL_RNN_nomm_43_2147132.err",
    "logs/advP_B_IQL_RNN_nomm_44_2147135.err",
]
ADVP_EXPOCOMM_STATIC = [
    "logs/advP_EC_st_nomm_42_2214582.err",
    "logs/advP_EC_st_nomm_43_2214584.err",
    "logs/advP_EC_st_nomm_44_2214586.err",
]
ADVP_HYBRIDCOMM_STATIC = [
    "logs/advP_HC_st_nomm_42_2213112.err",
    "logs/advP_HC_st_nomm_43_2213113.err",
    "logs/advP_HC_st_nomm_44_2213114.err",
]
ADVP_BIG_EXPOCOMM_STATIC = [
    "logs/advP_B_EC_st_nomm_42_2214587.err",
    "logs/advP_B_EC_st_nomm_43_2214588.err",
    "logs/advP_B_EC_st_nomm_44_2214589.err",
]
ADVP_BIG_HYBRIDCOMM_STATIC = [
    "logs/advP_B_HC_st_nomm_42_2213115.err",
    "logs/advP_B_HC_st_nomm_43_2213116.err",
    "logs/advP_B_HC_st_nomm_44_2213117.err",
]

ADV_PURSUIT = {"ExpoComm": ADVP_EXPOCOMM, "HybridComm": ADVP_HYBRIDCOMM, "IQL with RNN": ADVP_IQL_RNN}
ADV_PURSUIT_BIG = {"ExpoComm": ADVP_BIG_EXPOCOMM, "HybridComm": ADVP_BIG_HYBRIDCOMM, "IQL with RNN": ADVP_BIG_IQL_RNN}
ADV_PURSUIT_STATIC = {"ExpoComm Static": ADVP_EXPOCOMM_STATIC, "HybridComm Static": ADVP_HYBRIDCOMM_STATIC, "IQL with RNN": ADVP_IQL_RNN}
ADV_PURSUIT_BIG_STATIC = {"ExpoComm Static": ADVP_BIG_EXPOCOMM_STATIC, "HybridComm Static": ADVP_BIG_HYBRIDCOMM_STATIC, "IQL with RNN": ADVP_BIG_IQL_RNN}

## 1. Adversarial Pursuit

In [ ]:
plot_multi_group_performance_advP(ADV_PURSUIT, "Adversarial Pursuit Mean Test Return")

## 2. Adversarial Pursuit, Big

In [ ]:
plot_multi_group_performance_advP(ADV_PURSUIT_BIG, "Adversarial Pursuit Mean Test Return (Big)")

## 3. Adversarial Pursuit, Static

In [ ]:
plot_multi_group_performance_advP(ADV_PURSUIT_STATIC, "Adversarial Pursuit Mean Test Return (Static)")

## 4. Adversarial Pursuit, Big Static

In [ ]:
plot_multi_group_performance_advP(ADV_PURSUIT_BIG_STATIC, "Adversarial Pursuit Mean Test Return (Big, Static)")

# Battle

The plotting function below is the battle function from `reproduce_results.ipynb`.

In [ ]:
def exponential_moving_average(values, coefficient=0.85):
    values = np.atleast_1d(np.asarray(values, dtype=np.float64))
    if len(values) == 0:
        return np.array([])
    smoothed = []
    last = values[0]
    for point in values:
        smoothed_val = (last * coefficient) + (point * (1 - coefficient))
        smoothed.append(smoothed_val)
        last = smoothed_val
    return np.array(smoothed)

def parse_group_logs(file_paths):
    aggregated_data = defaultdict(list)
    t_env_pattern = re.compile(r"Recent Stats \| t_env:\s*(\d+)")
    test_return_pattern = re.compile(r"test_return_mean:\s*(-?\d+\.?\d*)")

    for path in file_paths:
        if not os.path.exists(path):
            print(f"Warning: File not found at {path}. Skipping.")
            continue
        current_t_env = None
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if "Recent Stats | t_env:" in line:
                    t_match = t_env_pattern.search(line)
                    if t_match:
                        current_t_env = int(t_match.group(1))
                if current_t_env is not None and "test_return_mean:" in line:
                    r_match = test_return_pattern.search(line)
                    if r_match:
                        aggregated_data[current_t_env].append(float(r_match.group(1)))
                        current_t_env = None

    sorted_timesteps = sorted(aggregated_data)
    means = [np.mean(aggregated_data[t]) for t in sorted_timesteps]
    return sorted_timesteps, np.array(means)

def plot_multi_group_performance_battle(groups_data, plot_title="Mean Test Return", smoothing_coeff=0.85):
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=(11, 6.5))
    colors = ["#FF0000", "#0000FF", "#137333"]

    for idx, (group_name, paths) in enumerate(groups_data.items()):
        steps, y_mean = parse_group_logs(paths)
        if not steps or len(y_mean) == 0:
            print(f"Skipping plot for group '{group_name}' due to missing data.")
            continue
        color = colors[idx % len(colors)]
        num_runs = len(paths)
        smoothed_y_mean = exponential_moving_average(y_mean, coefficient=smoothing_coeff)
        ax.plot(steps, smoothed_y_mean, color=color, linewidth=2.5,
                label=f"{group_name} ({num_runs} runs)")

    ax.set_title(plot_title, fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Timestep", fontsize=11, labelpad=10)
    ax.set_ylabel("Mean Test Return (EMA Smoothed, α=0.85)", fontsize=11, labelpad=10)
    ax.ticklabel_format(style="scientific", scilimits=(0, 0), axis="x")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(frameon=True, facecolor="white", edgecolor="none", loc="best")
    plt.tight_layout()
    plt.show()

In [ ]:
BATTLE_EXPOCOMM = [
    "logs/bat_ExpoComm_5M_original_rewards_1870522.err",
    "logs/bat_ExpoComm_5M_43_1871349.err",
    "logs/bat_ExpoComm_5M_44_1873123.err",
    "logs/bat_ExpoComm_5M_45_1874978.err",
    "logs/bat_ExpoComm_5M_46_1882050.err",
]
BATTLE_HYBRIDCOMM = [
    "logs/bat_HybridComm_5M_original_rewards_1870524.err",
    "logs/bat_HybridComm_5M_43_1871351.err",
    "logs/bat_HybridComm_5M_44_1874951.err",
    "logs/bat_HybridComm_5M_45_1874979.err",
    "logs/bat_HybridComm_5M_46_1882051.err",
]
BATTLE_IQL_RNN = [
    "logs/bat_IQL_RNN_5M_original_rewards_1870526.err",
    "logs/bat_IQL_RNN_5M_43_1871352.err",
    "logs/bat_IQL_RNN_5M_44_1874953.err",
    "logs/bat_IQL_RNN_5M_45_1874981.err",
    "logs/bat_IQL_RNN_5M_46_1882052.err",
]
BATTLE_60_EXPOCOMM = [
    "logs/bat_60_ExpoComm_5M_42_1872728.err",
    "logs/bat_60_ExpoComm_5M_43_1897904.err",
    "logs/bat_60_ExpoComm_5M_44_1887238.err",
    "logs/bat_60_ExpoComm_5M_45_1895082.err",
    "logs/bat_60_ExpoComm_5M_46_1897870.err",
]
BATTLE_60_HYBRIDCOMM = [
    "logs/bat_60_HybridComm_5M_42_1872729.err",
    "logs/bat_60_HybridComm_5M_43_1874997.err",
    "logs/bat_60_HybridComm_5M_44_1882096.err",
    "logs/bat_60_HybridComm_5M_45_1895084.err",
    "logs/bat_60_HybridComm_5M_46_1897871.err",
]
BATTLE_60_IQL_RNN = [
    "logs/bat_60_IQL_RNN_5M_42_1872730.err",
    "logs/bat_60_IQL_RNN_5M_43_1874999.err",
    "logs/bat_60_IQL_RNN_5M_44_1882097.err",
    "logs/bat_60_IQL_RNN_5M_45_1895086.err",
    "logs/bat_60_IQL_RNN_5M_46_1897859.err",
]
BATTLE_60_EXPOCOMM_N9 = [
    "logs/bat_60_ExpoComm_n9_42_2233055.err",
    "logs/bat_60_ExpoComm_n9_43_2233056.err",
]
BATTLE_60_HYBRIDCOMM_N9 = [
    "logs/bat_60_HybridComm_n9_42_2233057.err",
    "logs/bat_60_HybridComm_n9_43_2233058.err",
]
BATTLE_EXPOCOMM_STATIC = [
    "logs/bat_ExpoComm_static_42_2218606.err",
    "logs/bat_ExpoComm_static_43_2218607.err",
    "logs/bat_ExpoComm_static_44_2218609.err",
    "logs/bat_ExpoComm_static_45_2218610.err",
    "logs/bat_ExpoComm_static_46_2218611.err",
]
BATTLE_HYBRIDCOMM_STATIC = [
    "logs/bat_HybridComm_static_42_2090694.err",
    "logs/bat_HybridComm_static_43_2093588.err",
    "logs/bat_HybridComm_static_44_2167415.err",
    "logs/bat_HybridComm_static_45_2203240.err",
    "logs/bat_HybridComm_static_46_2203242.err",
]
BATTLE_60_EXPOCOMM_STATIC = [
    "logs/bat_60_ExpoComm_static_42_2219257.err",
    "logs/bat_60_ExpoComm_static_43_2219240.err",
    "logs/bat_60_ExpoComm_static_44_2219241.err",
    "logs/bat_60_ExpoComm_static_45_2219242.err",
    "logs/bat_60_ExpoComm_static_46_2219243.err",
]
BATTLE_60_HYBRIDCOMM_STATIC = [
    "logs/bat_60_HybridComm_static_42_2090703.err",
    "logs/bat_60_HybridComm_static_43_2093590.err",
    "logs/bat_60_HybridComm_static_44_2167417.err",
    "logs/bat_60_HybridComm_static_45_2203246.err",
    "logs/bat_60_HybridComm_static_46_2203248.err",
]

BATTLE = {"ExpoComm": BATTLE_EXPOCOMM, "HybridComm": BATTLE_HYBRIDCOMM, "IQL with RNN": BATTLE_IQL_RNN}
BATTLE_60 = {"ExpoComm": BATTLE_60_EXPOCOMM, "HybridComm": BATTLE_60_HYBRIDCOMM, "IQL with RNN": BATTLE_60_IQL_RNN}
BATTLE_60_N9 = {"ExpoComm n9": BATTLE_60_EXPOCOMM_N9, "HybridComm n9": BATTLE_60_HYBRIDCOMM_N9, "IQL with RNN": BATTLE_60_IQL_RNN}
BATTLE_STATIC = {"ExpoComm Static": BATTLE_EXPOCOMM_STATIC, "HybridComm Static": BATTLE_HYBRIDCOMM_STATIC, "IQL with RNN": BATTLE_IQL_RNN}
BATTLE_60_STATIC = {"ExpoComm Static": BATTLE_60_EXPOCOMM_STATIC, "HybridComm Static": BATTLE_60_HYBRIDCOMM_STATIC, "IQL with RNN": BATTLE_60_IQL_RNN}

## 5. Battle

In [ ]:
plot_multi_group_performance_battle(BATTLE, "Battle Mean Test Return")

## 6. Battle 60

In [ ]:
plot_multi_group_performance_battle(BATTLE_60, "Battle 60 Mean Test Return")

## 7. Battle 60, n9

In [ ]:
plot_multi_group_performance_battle(BATTLE_60_N9, "Battle 60 Mean Test Return (n9)")

## 8. Battle, Static

In [ ]:
plot_multi_group_performance_battle(BATTLE_STATIC, "Battle Mean Test Return (Static)")

## 9. Battle 60, Static

In [ ]:
plot_multi_group_performance_battle(BATTLE_60_STATIC, "Battle 60 Mean Test Return (Static)")